In [1]:
# ============================================================
# ML VALUATION EVALUATION FRAMEWORK - COMPLETE DEMO
# Saves all outputs and charts in /results folder
# Compatible with your project structure
# ============================================================
# Import modular evaluation functions
from evaluation.metrics import compute_metrics
from evaluation.statistical_tests import paired_t_test, bootstrap_ci
from evaluation.uncertainty import conformal_interval
from evaluation.segmentation import segment_evaluation
from evaluation.gating import deployment_decision
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats

# ============================================================
# 1️⃣ Confirm Working Directory
# ============================================================

print("Current Working Directory:")
print(os.getcwd())

# ============================================================
# 2️⃣ Load Dataset (from data/)
# ============================================================

data_path = os.path.join("data", "simulated_data.csv")
data = pd.read_csv(data_path)

print("\nDataset Loaded Successfully ✅")
print("Shape:", data.shape)

# ============================================================
# 3️⃣ Extract Variables
# ============================================================

y_true = data["true_value"]
y_champion = data["pred_champion"]
y_challenger = data["pred_challenger"]

# ============================================================
# 4️⃣ Define Metrics Function
# ============================================================

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    wmape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, mape, wmape, r2

# ============================================================
# 5️⃣ Compute Metrics
# ============================================================

metrics_champion = compute_metrics(y_true, y_champion)
metrics_challenger = compute_metrics(y_true, y_challenger)

print("\nChampion Metrics (MAE, RMSE, MAPE, WMAPE, R2):")
print(metrics_champion)

print("\nChallenger Metrics (MAE, RMSE, MAPE, WMAPE, R2):")
print(metrics_challenger)

# ============================================================
# 6️⃣ Statistical Test (Paired t-test)
# ============================================================

errors_champion = np.abs(y_true - y_champion)
errors_challenger = np.abs(y_true - y_challenger)

t_stat, p_value = stats.ttest_rel(errors_champion, errors_challenger)
print("\nPaired t-test p-value:", p_value)

# ============================================================
# 7️⃣ Segment Evaluation
# ============================================================

segment_results = (
    data.groupby("feature_2", as_index=False)
    .apply(lambda df: pd.Series({
        "MAE_Champion": mean_absolute_error(df["true_value"], df["pred_champion"]),
        "MAE_Challenger": mean_absolute_error(df["true_value"], df["pred_challenger"])
    }))
    .reset_index()
)

print("\nSegment Evaluation:")
print(segment_results)

# ============================================================
# 8️⃣ Conformal Coverage (Simple Approximation)
# ============================================================

residuals = y_true - y_challenger
q_hat = np.quantile(np.abs(residuals), 0.90)

lower_bound = y_challenger - q_hat
upper_bound = y_challenger + q_hat

coverage = np.mean((y_true >= lower_bound) & (y_true <= upper_bound))
print("\nConformal 90% Coverage:", coverage)

# ============================================================
# 9️⃣ Final Go/No-Go Decision
# ============================================================

if p_value < 0.05 and metrics_challenger[0] < metrics_champion[0]:
    decision = "GO ✅ - Challenger is statistically better."
else:
    decision = "NO-GO ❌ - Challenger needs improvement."

print("\nFinal Deployment Decision:")
print(decision)

# ============================================================
# 🔟 SAVE ALL OUTPUTS TO /results
# ============================================================

results_path = os.path.join("results")
os.makedirs(results_path, exist_ok=True)

print("\nSaving outputs to:", os.path.abspath(results_path))

# ---- Save Metrics ----
metrics_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "MAPE", "WMAPE", "R2"],
    "Champion": metrics_champion,
    "Challenger": metrics_challenger
})
metrics_df.to_csv(os.path.join(results_path, "evaluation_metrics.csv"), index=False)

# ---- Save Segment Results ----
segment_results.to_csv(os.path.join(results_path, "segment_analysis.csv"), index=False)

# ---- Save Final Decision & p-value & Coverage ----
with open(os.path.join(results_path, "final_decision.txt"), "w", encoding="utf-8") as f:
    f.write(f"Paired t-test p-value: {p_value:.5f}\n")
    f.write(f"Conformal Coverage: {coverage:.2f}\n")
    f.write(f"Decision: {decision}\n")

# ---- Save Charts ----
# MAE Comparison Chart
plt.figure(figsize=(6,4))
plt.bar(["Champion", "Challenger"], [metrics_champion[0], metrics_challenger[0]], color=["blue","orange"])
plt.title("MAE Comparison")
plt.ylabel("MAE")
plt.savefig(os.path.join(results_path, "mae_comparison.png"))
plt.close()

# Segment-wise MAE Comparison Chart
plt.figure(figsize=(10,5))
plt.bar(segment_results["feature_2"], segment_results["MAE_Champion"], alpha=0.6, label="Champion")
plt.bar(segment_results["feature_2"], segment_results["MAE_Challenger"], alpha=0.6, label="Challenger")
plt.title("Segment-wise MAE Comparison")
plt.xlabel("Segment (feature_2)")
plt.ylabel("MAE")
plt.legend()
plt.savefig(os.path.join(results_path, "segment_mae_comparison.png"))
plt.close()

print("\nAll outputs and charts successfully saved inside /results folder ✅")

Current Working Directory:
F:\GITHUB\ml-valuation-evaluation-framework

Dataset Loaded Successfully ✅
Shape: (100, 7)

Champion Metrics (MAE, RMSE, MAPE, WMAPE, R2):
(1340.2183520600051, np.float64(1709.577448974419), np.float64(2.663944578459208), np.float64(2.702085717899252), 0.9935589243214901)

Challenger Metrics (MAE, RMSE, MAPE, WMAPE, R2):
(1159.159004780524, np.float64(1484.0318582022055), np.float64(2.3706322120501078), np.float64(2.33704230864877), 0.9951463630361677)

Paired t-test p-value: 0.14312024344739505


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_17644\3555545117.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: pd.Series({



Segment Evaluation:
   index  feature_2  MAE_Champion  MAE_Challenger
0      0          1   1023.305863     1321.129073
1      1          2   2149.799960     1025.111266
2      2          3   1331.515612     1189.712631
3      3          4   1498.956306     1114.240990
4      4          5   1057.295837     1251.938277
5      5          6   1500.052356     1258.117375
6      6          7   1312.315840     1046.807432
7      7          8   1312.884161     1006.372606
8      8          9   1130.049334     1165.217583

Conformal 90% Coverage: 0.9

Final Deployment Decision:
NO-GO ❌ - Challenger needs improvement.

Saving outputs to: F:\GITHUB\ml-valuation-evaluation-framework\results

All outputs and charts successfully saved inside /results folder ✅
